# LoopFormer — `armenjeddi/LoopFormer-3block-8iterations`

Looped Transformer with 3 shared blocks and up to 8 recurrent iterations.  
The `steps` argument controls iteration schedule (list of floats that sum to 1).

In [1]:
# set HF cache
import os
os.environ["HF_HOME"] = "/tmp/hf_cache"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "armenjeddi/LoopFormer-3block-8iterations"
DEVICE = "cuda:7" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).to(DEVICE)

# GPTConfig uses n_layer; patch in the standard HF attribute that generate() needs
model.config.num_hidden_layers = model.config.n_layer

model.eval()
print(f"Loaded on {DEVICE}  |  dtype: {next(model.parameters()).dtype}")

/raid/s3/opengptx/behzad_shomali/modalities/olmes/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/raid/s3/opengptx/behzad_shomali/modalities/olmes/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/armenjeddi/LoopFormer-3block-8iterations:
- modeling_loopformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/a

number of parameters: 379.10M
Loaded on cuda:7  |  dtype: torch.bfloat16


## Text Generation

`steps` is a list of floats (the iteration schedule).  
The default is 8 equal steps `[1/8]*8`. You can use fewer steps (e.g. `[1/4]*4`) to trade quality for speed.

In [3]:
prompt = "The quick brown fox"

# Number of recurrent iterations (1–8). More iterations → better quality.
N_STEPS = 8
steps = [1 / N_STEPS] * N_STEPS

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        steps=steps,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated)

AttributeError: 'GPTConfig' object has no attribute 'num_hidden_layers'

## Compare different iteration counts

In [ ]:
prompt = "Once upon a time"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

for n in [1, 2, 4, 8]:
    steps = [1 / n] * n
    with torch.no_grad():
        out = model.generate(
            **inputs,
            steps=steps,
            max_new_tokens=60,
            do_sample=False,           # greedy for reproducibility
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"[{n} iteration(s)] {text}")
    print()